# Stage C 03s — E25/E50/E75/E100 milestone evaluation

This notebook evaluates an immutable adaptive trailblazer checkpoint. E25 and
E100 use the full analysis tier. E50 and E75 default to bounded held-out,
generation, memory, and context-anomaly diagnostics. Evaluations never write to
the training directory and do not pause the CURC continuation.


In [ ]:
# @title 1. Select the milestone
MILESTONE='e50' # @param ["e25", "e50", "e75", "e100"]
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC' # @param {type:"string"}
MODEL_PATH='' # @param {type:"string"}
RUN_FULL_INTERMEDIATE=False # @param {type:"boolean"}
GIT_REF='main' # @param {type:"string"}
TRUST_OWNED_CHECKPOINT=True

from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
root=Path(DRIVE_ROOT)
if MODEL_PATH:
    checkpoint=Path(MODEL_PATH)
elif MILESTONE=='e25':
    checkpoint=root/'runs/c19_v3_medium_adaptive_e25/latest.pt'
else:
    checkpoint=root/'runs/c20_v3_medium_adaptive_e100_increment/milestones'/MILESTONE/'model.pt'
if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
dataset=root/'stage_c_dataset/ordered_streams/nonoverlap_6mer_v1'
panels=root/'study/stage_c_ecoli_medium_deep_memory_v3/panels'
protocol=root/'study/stage_c_ecoli_medium_deep_memory_v3/protocol.json'
taxonomy=root/'stage_c_dataset/manifests/accession_manifest.parquet'
output=root/'runs/c20_v3_medium_adaptive_e100_increment/milestone_evaluations'/MILESTONE
output.mkdir(parents=True,exist_ok=True)
for path in (dataset,panels/'validation.json',protocol,taxonomy):
    if not path.exists(): raise FileNotFoundError(path)
print('Checkpoint:',checkpoint)


In [ ]:
# @title 2. Bootstrap the pinned evaluator
repo=Path('/content/SeqTrainer-milestone-evaluation')
if not repo.exists(): subprocess.run(['git','clone','https://github.com/Gonza10V/SeqTrainer.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',f'origin/{GIT_REF}'],check=True)
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
venv=Path('/content/seqtrainer-milestone-venv')
if not (venv/'bin/python').is_file():
    subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(venv)],check=True)
python=venv/'bin/python'
subprocess.run([str(python),'-m','pip','install','--quiet','--no-deps','-e',str(repo)],check=True)
if not TRUST_OWNED_CHECKPOINT: raise ValueError('Full-state Stage C checkpoint requires explicit trust')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'
def tool(name): return str(venv/'bin'/name)
stage_c_runner=tool('seqtrainer-titans-stage-c-colab-run')
def run(label,command):
    notebook_run=output/'notebook_runs'
    result=subprocess.run([stage_c_runner,'--run-dir',str(notebook_run),'--label',label,
      '--repo',str(repo),'--',*command],text=True,env=os.environ)
    if result.returncode:
        log=notebook_run/'logs'/f'{label}.log'
        raise RuntimeError(f'{label} failed; inspect {log}')


In [ ]:
# @title 3. Run the milestone tier
full=MILESTONE in {'e25','e100'} or RUN_FULL_INTERMEDIATE
limits=[] if full else ['--max-streams','4','--max-segments','256','--max-segments-per-accession','256']
run_id={'e25':'medium_adaptive_e25_analysis_v1','e50':'medium_adaptive_e50_analysis_v1',
 'e75':'medium_adaptive_e75_analysis_v1','e100':'medium_adaptive_e100_analysis_v1'}[MILESTONE]
amendment=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/amendments/adaptive_trailblazer_e100_v1.json'
amendment_args=[] if MILESTONE in {'e25','e100'} else ['--protocol-amendment',str(amendment)]
run('heldout',[tool('seqtrainer-titans-stage-c-evaluate'),'--dataset-dir',str(dataset),
 '--panel-manifest',str(panels/'validation.json'),'--run',f'adaptive={checkpoint}',
 '--output-dir',str(output/'evaluation'),'--split','val','--comparison-mode','partial',
 '--device','cuda','--resume','--checkpoint-every-segments','512','--progress-every-segments','16',
 '--protocol',str(protocol),*amendment_args,'--run-id',run_id,*limits])
run('memory_behavior',[tool('seqtrainer-titans-stage-c-memory-behavior'),'--checkpoint',str(checkpoint),
 '--output',str(output/'memory_behavior.json'),'--pairs','64' if full else '16','--device','cuda'])
trace_segments='512' if full else '128'
run('memory_trace',[tool('seqtrainer-titans-stage-c-memory-trace'),'--dataset-dir',str(dataset),
 '--panel-manifest',str(panels/'validation.json'),'--checkpoint',str(checkpoint),
 '--output-dir',str(output/'memory_trace'),'--split','val','--memory-mode','adaptive',
 '--max-streams','12' if full else '4','--max-segments',trace_segments,'--device','cuda'])
if not shutil.which('prodigal'):
    subprocess.run(['apt-get','update'],check=True); subprocess.run(['apt-get','install','-y','prodigal'],check=True)
run('generation',[tool('seqtrainer-titans-stage-c-generate'),'--dataset-dir',str(dataset),
 '--panel-manifest',str(panels/'validation.json'),'--checkpoint',str(checkpoint),
 '--output-dir',str(output/'generation_t0p6'),'--split','val','--species','Escherichia coli',
 '--taxonomy-manifest',str(taxonomy),
 '--prompts','4' if full else '2','--prompt-tokens','128','--new-tokens','1024' if full else '256',
 '--temperatures','0.6','--top-k','1024','--top-p','0.99','--device','cuda',
 '--memory-mode','adaptive','--prodigal',shutil.which('prodigal')])
print('Core milestone evaluation complete:',output)


In [ ]:
# @title 4. Run context anomaly and DNA-needle evaluation
registry=output/'context_registry'; inbox=registry/'inbox'; inbox.mkdir(parents=True,exist_ok=True)
source=inbox/'latest.pt'
if source.exists():
    def sha(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
        return h.hexdigest()
    if sha(source)!=sha(checkpoint):
        raise RuntimeError(f'Immutable context inbox already contains a different checkpoint: {source}')
else:
    partial=source.with_suffix('.pt.partial'); shutil.copy2(checkpoint,partial)
    def sha(path):
        h=hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
        return h.hexdigest()
    if sha(partial)!=sha(checkpoint): raise RuntimeError('context checkpoint staging checksum failed')
    os.replace(partial,source)
inspect=r'''import json,sys,torch
p=torch.load(sys.argv[1],map_location='cpu',weights_only=False); t=p.get('trainer_state',{})
print(json.dumps({'optimizer_step':int(t.get('optimizer_step',-1)),
 'processed_bases':int(t.get('processed_bases',-1)),'milestone':sys.argv[2],
 'code_commit':p.get('code_commit'),'dataset_fingerprint':p.get('dataset_fingerprint')}))'''
metadata_payload=json.loads(subprocess.check_output([str(python),'-c',inspect,str(source),MILESTONE],text=True,env=os.environ))
metadata=output/'context_metadata.json'
metadata.write_text(json.dumps(metadata_payload,indent=2,sort_keys=True)+'\n')
context=tool('seqtrainer-titans-stage-c-context-eval')
run('context_stage',[context,'stage','--source',str(source),'--registry',str(registry),
 '--metadata-json',str(metadata),'--trust-owned-checkpoint'])
models=sorted((registry/'models').iterdir())
if not models: raise RuntimeError('context model registry is empty')
base_context=[context,'run','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'validation.json'),
 '--model-dir',str(models[-1]),'--registry',str(registry),'--split','val',
 '--code-commit',commit,'--device','cuda','--trust-owned-checkpoint']
run('context_smoke',[*base_context,'--mode','smoke'])
if full: run('context_full',[*base_context,'--mode','full'])
run('context_catalog',[context,'catalog','--registry',str(registry)])
print('Context/anomaly evaluation complete:',registry)
